# CodeMix Scribe — Gemma correction server (Kaggle)

Runs the fine-tuned Gemma LoRA adapter on a free Kaggle GPU and exposes `POST /correct`
over a public **cloudflared** URL that the laptop's ai-service calls.

**Before running:**
1. *Settings → Accelerator:* **GPU T4 x2** or **T4 x1** (only one GPU is used). *Settings → Internet:* **On**.
2. *Add Input:* your Kaggle Dataset that contains the adapter folder
   (`adapter_config.json`, `adapter_model.safetensors`, tokenizer files).
3. *Run All.* The last-but-one cell prints a line like
   `GEMMA URL: https://something.trycloudflare.com` — paste that into the CodeMix Scribe Settings page.

The base model is downloaded from Hugging Face automatically; only the adapter comes from your dataset.

In [ ]:
# 1) Install packages (takes ~2-4 minutes)
!pip install -q --upgrade unsloth unsloth_zoo
!pip install -q fastapi "uvicorn[standard]"

In [ ]:
# 2) Settings
import glob, os, sys
# Use ONE GPU only (Kaggle "T4 x2" splits the model across both, which breaks sampling),
# and turn off Unsloth's torch.compile / CUDA graphs. Must happen before torch is imported.
assert "torch" not in sys.modules, "Run -> Restart & clear outputs, then run the cells again from the top."
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"

BASE_MODEL = "unsloth/gemma-4-e4b-it"   # loaded in 4-bit
GEMMA_TOKEN = ""                         # optional: set a password here and the same value as GEMMA_TOKEN in ai-service/.env

found = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
assert found, "Adapter not found. Click 'Add Input' and attach your adapter dataset."
ADAPTER_PATH = os.path.dirname(found[0])
print("Adapter:", ADAPTER_PATH)
print(os.listdir(ADAPTER_PATH))

In [ ]:
# 3) Load base model in 4-bit + attach the LoRA adapter
from unsloth import FastLanguageModel
from peft import PeftModel
import torch

model, processor = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=512,
    load_in_4bit=True,
    dtype=torch.float16,      # T4 has no native bfloat16
    device_map={"": 0},       # everything on GPU 0
)
model = PeftModel.from_pretrained(model, ADAPTER_PATH)
FastLanguageModel.for_inference(model)
tokenizer = getattr(processor, "tokenizer", processor)   # Gemma 4 returns a processor
print("Loaded on", torch.cuda.get_device_name(0))

In [ ]:
# 4) Correction function — same prompt, chunking and generation settings the adapter was trained with
import re, threading
from transformers import StoppingCriteria, StoppingCriteriaList

# Gemma sometimes writes a malformed end marker (e.g. "<end_of_of_turn>") and keeps going.
# Stop on ANY chat-control marker and cut the answer there.
MARKER = re.compile(r"<\s*/?\s*(?:end_of|start_of|eos)[^>]*>", re.I)

PROMPT = (
    "<start_of_turn>user\n"
    "Fix ASR errors. Preserve code-mixed script (keep English words in Latin script, "
    "native language words in their own script).\n"
    "{chunk}<end_of_turn>\n<start_of_turn>model\n"
)
gen_lock = threading.Lock()


class StopOnSubstring(StoppingCriteria):
    def __init__(self, stop, prompt_len):
        self.stop, self.prompt_len = stop, prompt_len
    def __call__(self, input_ids, scores, **kwargs):
        return bool(MARKER.search(tokenizer.decode(input_ids[0][self.prompt_len:], skip_special_tokens=False)))


def chunk_by_words(text, size=55):
    w = text.split()
    return [" ".join(w[i:i + size]) for i in range(0, len(w), size)]


def correct_chunk(chunk):
    prompt = PROMPT.format(chunk=chunk)
    inputs = processor(text=prompt, return_tensors="pt").to(model.device)
    plen = inputs["input_ids"].shape[1]
    args = dict(
        max_new_tokens=200,
        stopping_criteria=StoppingCriteriaList([StopOnSubstring("<end_of_turn>", plen)]),
        pad_token_id=tokenizer.pad_token_id,
    )
    with gen_lock, torch.inference_mode():
        try:
            out = model.generate(**inputs, do_sample=True, temperature=0.3, **args)
        except RuntimeError as e:   # sampling RNG error on some GPU setups -> greedy decoding
            print("sampling failed, using greedy decoding:", str(e)[:120])
            out = model.generate(**inputs, do_sample=False, **args)
    text = tokenizer.decode(out[0][plen:], skip_special_tokens=False)
    m = MARKER.search(text)
    return (text[:m.start()] if m else text).strip()


def correct(text):
    if not text.strip():
        return text
    return " ".join(correct_chunk(c) for c in chunk_by_words(text))

In [ ]:
# 5) Test cell: correct a sample code-mixed sentence
import time
sample = "Doctor எனக்கு two weeks ஆ cough இருக்கு night la fever um varudhu"
t = time.time()
print("IN :", sample)
print("OUT:", correct(sample))
print(f"({time.time() - t:.1f}s)")

In [ ]:
# 6) FastAPI server on port 8000, in a background thread
import uvicorn
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel

app = FastAPI(title="CodeMix Scribe Gemma")

class CorrectIn(BaseModel):
    text: str

def check_token(authorization):
    if GEMMA_TOKEN and authorization != f"Bearer {GEMMA_TOKEN}":
        raise HTTPException(401, "bad token")

@app.get("/health")
def health(authorization: str | None = Header(default=None)):
    check_token(authorization)
    return {"status": "ok", "gpu": torch.cuda.get_device_name(0)}

@app.post("/correct")
def correct_endpoint(body: CorrectIn, authorization: str | None = Header(default=None)):
    check_token(authorization)
    return {"corrected": correct(body.text)}

server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(3)
import requests
print(requests.get("http://localhost:8000/health", headers={"Authorization": f"Bearer {GEMMA_TOKEN}"}).json())

In [ ]:
# 7) Public URL with a cloudflared quick tunnel
import re, subprocess
if not os.path.exists("cloudflared"):
    !wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x cloudflared

tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--no-autoupdate", "--url", "http://localhost:8000"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
GEMMA_PUBLIC_URL = None
for line in tunnel.stdout:
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if m:
        GEMMA_PUBLIC_URL = m.group(0)
        break

print("=" * 70)
print("GEMMA URL:", GEMMA_PUBLIC_URL)
print("Paste it in CodeMix Scribe -> Settings (or GEMMA_URL in ai-service/.env)")
print("=" * 70)

In [ ]:
# 8) Keep-alive: leave this running during the demo (stop it with the ■ button when done).
#    Prints a heartbeat every 5 minutes so you can see the server is still up.
threading.Thread(target=lambda: [l for l in tunnel.stdout], daemon=True).start()  # drain tunnel logs
while True:
    try:
        ok = requests.get("http://localhost:8000/health", headers={"Authorization": f"Bearer {GEMMA_TOKEN}"}, timeout=10).ok
    except Exception:
        ok = False
    print(time.strftime("%H:%M:%S"), "server ok" if ok else "server DOWN", GEMMA_PUBLIC_URL)
    time.sleep(300)